In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import hydra

from genpp import BASE_DIR
from genpp.configs import register_resolvers

In [ ]:
register_resolvers()

# Test if Model works in general

In [ ]:
with hydra.initialize_config_dir(version_base=None, config_dir=str(BASE_DIR / "configs")):
    cfg = hydra.compose(config_name="base_fmunet")

In [ ]:
datamodule = hydra.utils.instantiate(cfg.data.module)
datamodule.prepare_data()
datamodule.setup("fit")
dataloader = datamodule.train_dataloader()

In [ ]:
cfg.model.n_samples = 16  # for testing purposes only

In [ ]:
model = hydra.utils.instantiate(
    cfg.model, rescaler=datamodule.y_reverseModules if cfg.model.use_rescaler else None
)

In [ ]:
trainer = hydra.utils.instantiate(
    cfg.trainer,
    logger=None,
    detect_anomaly=True,
    accelerator="cpu",
    max_epochs=10,
    overfit_batches=10,
    check_val_every_n_epoch=5,
)
trainer.fit(model, datamodule=datamodule)

## Try to load the model from checkpoint

In [ ]:
import os
from pathlib import Path

from genpp.models.fm.base import FlowMatchingModel

cwd = Path(os.getcwd())
checkpoint_path = list((cwd / "lightning_logs" / "version_1" / "checkpoints").glob("*.ckpt"))[0]
checkpoint_path

In [ ]:
mod_new = FlowMatchingModel.load_from_checkpoint(checkpoint_path)